# 🤖🗣️ 演示：SmolVLA（视觉-语言-动作模型）

前面的 ACT / Diffusion Policy 都是**针对单个任务**从示范学出来的策略。
**SmolVLA** 是一个 **VLA（Vision-Language-Action）**模型：在一个**预训练的视觉语言模型（VLM）**之上接一个**动作专家**，
输入是 **图像 + 机器人状态 + 一句自然语言指令**，输出是动作块。**语言指令**让同一个模型能做不同的事——这就是 VLA 的 “L”。

| | ACT / DP | **SmolVLA** |
|---|---|---|
| 输入 | 图像 + 状态 | 图像 + 状态 **+ 语言指令** |
| 骨干 | 从零训的小网络 | **预训练 VLM**（SmolVLM2-500M）+ 动作专家 |
| 通用性 | 单任务 | **多任务 / 泛化**（换句话就换任务）|
| 本 demo | 自己训练 | **直接用官方预训练 `lerobot/smolvla_base`** |

本 demo 不训练，直接加载预训练模型、喂一帧观测 + 语言指令，看它输出动作，并演示**换指令动作就变**。

## 0. 环境安装

SmolVLA 依赖 **transformers**（视觉语言模型）。在 ACT/DP 的环境上再装 SmolVLA 这个 extra 即可：

```powershell
conda activate lerobot
pip install "lerobot[smolvla]"      # 装 transformers / accelerate / num2words
```

> 第一次运行会自动下载预训练模型 `lerobot/smolvla_base`（含 SmolVLM2-500M 骨干，约 1 GB）。**推理很轻**（约 1 GB 显存），8 GB 显卡毫无压力。

In [ ]:
import torch
from PIL import Image
from IPython.display import display

from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_pre_post_processors

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("设备:", device)

## 1. 加载预训练 SmolVLA

`from_pretrained` 会下载并加载 `lerobot/smolvla_base`；`make_pre_post_processors` 加载**模型自带**的归一化统计量（注意第二个参数传的是模型 id，不是数据集）。

In [ ]:
model_id = "lerobot/smolvla_base"
model = SmolVLAPolicy.from_pretrained(model_id).to(device)
model.eval()
preprocess, postprocess = make_pre_post_processors(model.config, model_id)

print("参数量:", f"{sum(p.numel() for p in model.parameters()):,}")
print("chunk_size:", model.config.chunk_size, "| n_obs_steps:", model.config.n_obs_steps)
print("期望的相机键:", list(model.config.image_features))
print("VLM 骨干:", model.config.vlm_model_name)

## 2. 模型结构

SmolVLA $=$ **预训练 VLM（SmolVLM）** $+$ **动作专家**。VLM 负责把图像和语言编码成理解，动作专家据此生成动作。

In [ ]:
def count(m):
    return sum(p.numel() for p in m.parameters())

print(f"总参数: {count(model):,}\n")
print(f"{'顶层子模块':<22}{'参数量':>16}")
print("-" * 40)
for name, child in model.named_children():
    if count(child):
        print(f"{name:<22}{count(child):>16,}")

## 3. 准备一帧观测（图像 + 状态 + 语言）

从 `svla_so101_pickplace` 取一帧来喂给模型。

⚠️ **相机键要对齐**：`smolvla_base` 期望 `observation.images.camera1/2/3`，而这个数据集是 `up/side`——
我们把 `up→camera1`、`side→camera2`（第 3 路缺失会被模型自动 mask 掉）。
> 注：这里的相机映射只是**演示流程**用；相机语义未必和预训练时完全一致，动作数值仅供观察管线是否跑通。

In [ ]:
ds = LeRobotDataset("lerobot/svla_so101_pickplace", video_backend="pyav")
s = ds[0]

def to_pil(t):
    a = (t.numpy().transpose(1, 2, 0) * 255).clip(0, 255).astype("uint8")
    return Image.fromarray(a)

print("模型看到的画面（up / side）：")
display(to_pil(s["observation.images.up"]).resize((240, 180)))
display(to_pil(s["observation.images.side"]).resize((240, 180)))

def make_obs(task_text):
    return {
        "observation.images.camera1": s["observation.images.up"].unsqueeze(0).to(device),
        "observation.images.camera2": s["observation.images.side"].unsqueeze(0).to(device),
        "observation.state": s["observation.state"].unsqueeze(0).to(device),
        "task": task_text,          # ← 自然语言指令（VLA 的 “L”）
    }

print("\n数据集自带的任务描述:", s["task"])

## 4. 推理：看图 + 读指令 → 输出动作

`select_action` 内部：图像过 VLM、语言 token 化并注入、动作专家生成动作块；返回当前这一步的 6 维关节动作。

In [ ]:
import time
task = s["task"]
model.reset()
t0 = time.perf_counter()
with torch.no_grad():
    action = postprocess(model.select_action(preprocess(make_obs(task))))
print(f"指令: 「{task}」")
print("输出动作 (6 关节):", action[0].cpu().numpy().round(3))
print(f"推理耗时: {time.perf_counter() - t0:.2f}s | 显存峰值: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

## 5. VLA 的关键：换一句语言指令，动作就变

**同一帧图像、同一个状态**，只改**语言指令**，看模型输出的动作如何不同——这正是 VLA「用语言指挥」的能力。

In [ ]:
instructions = [
    "pick up the pink lego brick",
    "put the brick into the box",
    "move the arm to the left",
    "open the gripper",
]
print(f"{'语言指令':<34}{'输出动作（6 关节）'}")
print("-" * 80)
for ins in instructions:
    model.reset()
    with torch.no_grad():
        a = postprocess(model.select_action(preprocess(make_obs(ins))))
    vals = "  ".join(f"{x:6.2f}" for x in a[0].cpu().numpy())
    print(f"{ins:<34}{vals}")
print("\n同样的画面，不同指令 → 不同动作：模型确实在'读'语言。")

## 小结：三种策略对比

| | ACT | Diffusion Policy | **SmolVLA** |
|---|---|---|---|
| 生成方式 | 直接回归 | 迭代去噪 | VLM $+$ 动作专家（flow matching）|
| 语言指令 | ❌ | ❌ | ✅ **核心输入** |
| 预训练 | 视觉骨干 | 视觉骨干 | ✅ **整个 VLM 预训练**，可零/少样本泛化 |
| 参数量 | ~52 M | ~263 M | ~450 M |
| 典型用法 | 单任务从示范训 | 单任务从示范训 | 拿预训练权重，少量数据微调到新任务 |

**要点**：SmolVLA 把「机器人策略」变成了一个**能听懂话的通用模型**——同一权重，换句指令就能做不同的事，还能用少量示范快速微调到你自己的任务。

### 想进一步做什么？
- **微调到自己的任务**：用 `lerobot-train --policy.path=lerobot/smolvla_base --dataset.repo_id=<你的数据>` 在少量示范上微调。
- **真机部署**：见 `snippets/ch5/02_using_smolvla.py`（连接 SO-101，实时看图 + 给指令 → 执行）。